# Heart Disease Prediction System


## About This Notebook

This notebook is the Heart Disease version of the
`Titanic_Passenger_Survival_Recommended.ipynb` notebook. It follows the **same
end-to-end machine learning workflow** (import libraries -> load data ->
understand -> pre-process -> feature selection -> encode -> split -> train ->
save -> test -> evaluate -> take user input -> predict -> improve -> deploy),
but applied to a real medical dataset instead of the Titanic toy sample.

Two things are done more carefully than in the Titanic notebook, because this
dataset is real data rather than a clean toy sample:

1. **All pre-processing lives inside a scikit-learn `Pipeline`.** The imputer,
   the scaler and the one-hot encoder are fitted on the **training split only**,
   so no information from the test set can leak into training.
2. **The pipeline and the model are saved together in one `.pkl` file.** The
   Streamlit app therefore only has to load one object; it cannot accidentally
   scale or encode the user's input differently from the way the model was
   trained.


## SU Meta Description
<p>
A beginner-friendly Heart Disease Prediction System built on the UCI Cleveland
heart disease dataset. A Support Vector Classifier uses four clinical attributes
to predict whether heart disease is indicated.
</p>

## Introduction

##### **Definition** A binary classification ML task that predicts whether heart disease is indicated (1) or not indicated (0) for a patient, using a small set of clinical attributes.
##### **Application** It is used here for learning data science and for understanding which clinical signs (chest pain type, exercise response, etc.) carry the most information about heart disease.

## Dataset Source

| | |
| --- | --- |
| **Name** | Heart Disease (Cleveland processed subset) |
| **Source** | UCI Machine Learning Repository, dataset #45 |
| **URL** | https://archive.ics.uci.edu/dataset/45/heart+disease |
| **Donors** | Andras Janosi (Hungarian Institute of Cardiology), William Steinbrunn (University Hospital, Zurich), Matthias Pfisterer (University Hospital, Basel), Robert Detrano (V.A. Medical Center, Long Beach / Cleveland Clinic Foundation) |
| **Local copy** | `heart.csv` - 303 rows, 14 columns, saved exactly as downloaded from UCI (missing values appear as `NaN`) |

The original database has 76 raw attributes, but every published experiment uses
the same **14-attribute processed Cleveland subset**, which is what `heart.csv`
contains. It is the only subset with a fully documented target variable, which
is why the assignment prefers it.

##  Input and Output

###  Inputs (the FOUR selected features / attributes)

| **Input Attribute** | **Meaning** | **Possible Values** |
| ------------ | -------------------------------- | ------------------- |
| **cp**      | Chest pain type                   | 1 = typical angina, 2 = atypical angina, 3 = non-anginal pain, 4 = asymptomatic |
| **thalach** | Maximum heart rate achieved during exercise | roughly 70-205 (beats per minute) |
| **exang**   | Exercise-induced angina           | 0 = no, 1 = yes |
| **oldpeak** | ST depression induced by exercise relative to rest | roughly 0.0-6.2 |

### Output (Prediction)

| **Output Attribute** | **Possible Values** |
| -------------------- | ------------------- |
| **target** (derived from the original `num`) | 0 = no disease indicated, 1 = disease indicated |

The original UCI target column is called **`num`** and holds values 0-4, where 0
means no disease and 1, 2, 3 and 4 are increasing degrees of vessel narrowing.
As the assignment requires, `num` is converted into a **binary** target:
`target = 0` when `num == 0`, and `target = 1` when `num` is 1, 2, 3 or 4.


##  Table of Content
- Step 1: Import Libraries
- Step 2: Load the Dataset (`heart.csv`)
- Step 3: Understand and Pre-process the Data
  - Step 3.1: Understand the Data
  - Step 3.2: Create the Binary Target from `num`
  - Step 3.3: Pre-process Missing Values (plan)
- Step 4: Feature Selection (exactly FOUR input attributes)
- Step 5: Encoding and Scaling (built as a leakage-free Pipeline)
  - Step 5.1: Which column needs which treatment
  - Step 5.2: Build the pre-processing ColumnTransformer
- Step 6: Execute the Training Phase
  - Step 6.1: Split into Training Data and Testing Data
  - Step 6.2: Input Vectors and Labels of the Training Data
  - Step 6.3: Train the Support Vector Classifier (inside the Pipeline)
  - Step 6.4: Save the Trained Pipeline
- Step 7: Execute the Testing Phase
  - Step 7.1: Input Vectors and Labels of the Testing Data
  - Step 7.2: Load the Saved Pipeline
  - Step 7.3: Make Predictions on the Testing Data
  - Step 7.4: Calculate the Accuracy Score
  - Step 7.5: Detailed Metrics (Precision, Recall, F1, Confusion Matrix)
- Step 8: Execute the Application Phase
  - Step 8.1: Take Input from the User
  - Step 8.2: Convert the User Input into a Feature Vector
  - Step 8.3: Load the Saved Pipeline
  - Step 8.4: Model Prediction
- Step 9: Execute the Feedback Phase
  - Step 9.1: Collect Feedback
  - Step 9.2: List Potential Improvements
  - Step 9.3: Improve the Model Based on Feedback
- Step 10: Cloud Deployment Readiness
- Conclusion


## Code - Heart Disease Prediction


###  Step 1: Import Libraries


In [1]:
"""
Purpose of this code:
----------------------
This cell imports every library the notebook needs: tools for handling data,
for splitting it into training and testing sets, for pre-processing it
(imputing, scaling, encoding), for training a Support Vector Machine, and for
measuring how good the trained model is.
"""

# Numerical computing library (arrays, math operations)
import numpy as np

# Data manipulation library (DataFrames, reading/writing CSV files)
import pandas as pd

# Pickle is used to save and load the trained model to/from a file
import pickle

# Splits the dataset into a training part and a testing part
from sklearn.model_selection import train_test_split

# SimpleImputer fills in missing values
from sklearn.impute import SimpleImputer

# StandardScaler puts numeric columns on the same scale (important for SVC);
# OneHotEncoder turns a categorical code into separate 0/1 columns
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ColumnTransformer applies different treatment to different columns
from sklearn.compose import ColumnTransformer

# Pipeline chains the pre-processing and the model into one single object
from sklearn.pipeline import Pipeline

# The Support Vector Machine classifier (same model family as the Titanic notebook)
from sklearn import svm

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Show every row and column instead of hiding some with "..."
pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")


Libraries imported successfully.


### Step 2: Load the Dataset (`heart.csv`)

`heart.csv` is the **processed Cleveland subset** of the UCI Heart Disease
dataset (https://archive.ics.uci.edu/dataset/45/heart+disease), saved into this
folder exactly as it was downloaded. No values were edited; missing entries are
left as `NaN`, which is how UCI exports them.


In [2]:
"""
Purpose of this section:
-------------------------
Load the local copy of the UCI Cleveland heart disease data into a pandas
DataFrame and look at the first few rows of raw data.
"""

# Read the CSV file that sits next to this notebook
heart_data = pd.read_csv("heart.csv")

print("\n\nHeart Disease Data (first 10 rows):")
print("===================================\n")
print(heart_data.head(10))




Heart Disease Data (first 10 rows):

   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   
5   56    1   2       120   236    0        0      178      0      0.8      1   
6   62    0   4       140   268    0        2      160      0      3.6      3   
7   57    0   4       120   354    0        0      163      1      0.6      1   
8   63    1   4       130   254    0        2      147      0      1.4      2   
9   53    1   4       140   203    1        2      155      1      3.1      3   

    ca  thal  num  
0  0.0   6.0    0  
1  3.0   3.0    2  
2  2.0   

### Step 3: Understand and Pre-process the Data


####  Step 3.1: Understand the Data


In [3]:
"""
Purpose of this section:
-------------------------
Get a feel for the dataset before touching it:
1. How many rows (patients) and columns (attributes) are there?
2. What are the column names?
3. Which columns have missing values?
4. How is the original target column `num` distributed?
"""

print("\n\nShape of the Dataset (rows, columns):", heart_data.shape)
print("==============================================\n")

print("\n\nAttributes in the Dataset:")
print("==========================\n")
print(list(heart_data.columns))

print("\n\nMissing Values per Attribute:")
print("=============================\n")
print(heart_data.isna().sum())

print("\n\nOriginal Target `num` (0 = no disease, 1-4 = degrees of disease):")
print("=================================================================\n")
print(heart_data["num"].value_counts().sort_index())

print("\n\nBasic Statistics of the Numeric Attributes:")
print("===========================================\n")
print(heart_data.describe())




Shape of the Dataset (rows, columns): (303, 14)



Attributes in the Dataset:

['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']


Missing Values per Attribute:

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
num         0
dtype: int64


Original Target `num` (0 = no disease, 1-4 = degrees of disease):

num
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64


Basic Statistics of the Numeric Attributes:

              age         sex          cp    trestbps        chol         fbs  \
count  303.000000  303.000000  303.000000  303.000000  303.000000  303.000000   
mean    54.438944    0.679868    3.158416  131.689769  246.693069    0.148515   
std      9.038662    0.467299    0.960126   17.599748   51.776918    0.356198   
min     29.000000    0.000000    1.000000 

#### Step 3.2: Create the Binary Target from `num`

The assignment requires the original UCI target `num` to be used, converted into
a **binary** classification target:

* `num == 0`  ->  **target = 0** (no disease indicated)
* `num` in 1, 2, 3, 4  ->  **target = 1** (disease indicated)

This is the standard way the Cleveland subset is used in the literature.


In [4]:
"""
Purpose of this section:
-------------------------
Convert the original 0-4 column `num` into a 0/1 column called `target`.
The expression (heart_data["num"] > 0) gives True/False, and .astype(int)
turns True into 1 and False into 0.
"""

heart_data["target"] = (heart_data["num"] > 0).astype(int)

print("\n\nOriginal `num` vs. the new binary `target`:")
print("===========================================\n")
print(heart_data[["num", "target"]].head(10))

print("\n\nBinary Target Distribution:")
print("===========================\n")
print(heart_data["target"].value_counts().sort_index())
print("\n0 = no disease indicated, 1 = disease indicated")
print("The two classes are close to balanced, so plain accuracy is a fair measure here.")




Original `num` vs. the new binary `target`:

   num  target
0    0       0
1    2       1
2    1       1
3    0       0
4    0       0
5    0       0
6    3       1
7    0       0
8    2       1
9    1       1


Binary Target Distribution:

target
0    164
1    139
Name: count, dtype: int64

0 = no disease indicated, 1 = disease indicated
The two classes are close to balanced, so plain accuracy is a fair measure here.


#### Step 3.3: Pre-process Missing Values (plan)

Step 3.1 showed that only two columns have missing values: `ca` (4 missing) and
`thal` (2 missing). Neither of them is one of the four attributes selected in
Step 4, so the four columns actually used for training are complete.

A missing-value step is still included, for two reasons:

1. It keeps the workflow correct if the selected features are ever changed.
2. It protects the deployed app: if a value ever arrives missing at prediction
   time, the pipeline fills it in instead of crashing.

**Important:** the imputer is **not** applied here. It is placed inside the
Pipeline in Step 5 and fitted in Step 6 on the **training split only**, so that
no information from the test set leaks into training.


In [5]:
"""
Purpose of this section:
-------------------------
Confirm which columns have missing values and confirm that the four attributes
selected in the next step contain no missing values at all.
"""

missing_counts = heart_data.isna().sum()

print("\n\nColumns That Contain Missing Values:")
print("====================================\n")
print(missing_counts[missing_counts > 0])

print("\n\nNo pre-processing is applied at this point on purpose.")
print("The imputer, the scaler and the encoder are all built into the Pipeline")
print("in Step 5 and fitted on the TRAINING data only in Step 6 (no data leakage).")




Columns That Contain Missing Values:

ca      4
thal    2
dtype: int64


No pre-processing is applied at this point on purpose.
The imputer, the scaler and the encoder are all built into the Pipeline
in Step 5 and fitted on the TRAINING data only in Step 6 (no data leakage).


#### Step 4: Feature Selection (exactly FOUR input attributes)

The assignment requires **exactly four** input attributes. The four chosen are:

| # | Attribute | Type | Why it was selected |
| - | --------- | ---- | ------------------- |
| 1 | **cp** (chest pain type) | categorical (1-4) | Chest pain is the primary presenting symptom of coronary artery disease. In this dataset, "asymptomatic" chest pain (`cp = 4`) is the single strongest signal of disease, which is a well-known and clinically meaningful finding. |
| 2 | **thalach** (maximum heart rate achieved) | numeric | How high the heart rate can climb during a stress test measures how well the heart copes with exertion. A low maximum heart rate indicates reduced cardiac reserve. |
| 3 | **exang** (exercise-induced angina) | binary (0/1) | Chest pain that appears *during exercise* is a classic sign that the heart muscle is not getting enough blood under load. |
| 4 | **oldpeak** (ST depression induced by exercise) | numeric | A direct ECG measurement of how far the ST segment drops during exercise - an objective, machine-measured indicator of reduced blood flow. |

**Why these four and not others:**

* They cover **four different kinds of evidence**: a reported symptom (`cp`),
  a performance measurement (`thalach`), an observed symptom under load
  (`exang`) and an objective instrument reading (`oldpeak`). Features that
  repeat the same information add little.
* They are all **recorded during a single exercise stress test**, so a user of
  the app can realistically supply all four from one report.
* They are all **complete** in the Cleveland subset (no missing values).
* `ca` and `thal`, although also strong predictors, were deliberately left out:
  they come from an invasive fluoroscopy / thallium scan, they are the only two
  columns with missing data, and they are much harder for a non-specialist user
  of the app to supply.


In [6]:
"""
Purpose of this section:
-------------------------
Keep exactly the four selected input attributes plus the binary target, and
confirm that the feature count is really four.
"""

# The FOUR selected input attributes
SELECTED_FEATURES = ["cp", "thalach", "exang", "oldpeak"]

# The name of the binary output column created in Step 3.2
TARGET_COLUMN = "target"

# X holds the inputs, y holds the output we want to predict
X = heart_data[SELECTED_FEATURES]
y = heart_data[TARGET_COLUMN]

print("\n\nSelected Input Attributes:", SELECTED_FEATURES)
print("Number of input attributes used:", len(SELECTED_FEATURES))
assert len(SELECTED_FEATURES) == 4, "The assignment requires exactly FOUR input attributes."

print("\n\nInput Features X (first 10 rows):")
print("=================================\n")
print(X.head(10))

print("\n\nOutput Target y (first 10 values):")
print("==================================\n")
print(y.head(10).to_list())




Selected Input Attributes: ['cp', 'thalach', 'exang', 'oldpeak']
Number of input attributes used: 4


Input Features X (first 10 rows):

   cp  thalach  exang  oldpeak
0   1      150      0      2.3
1   4      108      1      1.5
2   4      129      1      2.6
3   3      187      0      3.5
4   2      172      0      1.4
5   2      178      0      0.8
6   4      160      0      3.6
7   4      163      1      0.6
8   4      147      0      1.4
9   4      155      1      3.1


Output Target y (first 10 values):

[0, 1, 1, 0, 0, 0, 1, 0, 1, 1]


### Step 5: Encoding and Scaling (built as a leakage-free Pipeline)


#### Step 5.1: Which column needs which treatment

The Titanic notebook used a `LabelEncoder` on every column, because every Titanic
feature was a text category. Here the four features are of two different kinds,
so they need two different treatments:

| Attribute | Kind | Treatment | Why |
| --------- | ---- | --------- | --- |
| `thalach` | numeric | median imputer + **StandardScaler** | SVC measures distances between points, so a feature measured in the hundreds (heart rate) would completely drown out a feature measured in units (`oldpeak`) unless both are rescaled. |
| `oldpeak` | numeric | median imputer + **StandardScaler** | same reason |
| `cp` | categorical, 4 codes | most-frequent imputer + **OneHotEncoder** | The codes 1, 2, 3, 4 are *names*, not quantities - "type 4" is not "twice type 2". One-hot encoding turns the column into four 0/1 columns so the model does not invent a fake ordering. |
| `exang` | binary 0/1 | most-frequent imputer + **OneHotEncoder** | Already 0/1, so one-hot encoding is harmless, and routing it through the same branch keeps the code short and consistent. |

Note that scaling is applied **only** to the numeric columns; the one-hot columns
are already 0/1 and do not need it.


#### Step 5.2: Build the pre-processing ColumnTransformer

A `ColumnTransformer` is the object that applies different steps to different
columns. It is **built** here but **not fitted** here - fitting happens in
Step 6.3 on the training split only. That is exactly what prevents data leakage.


In [7]:
"""
Purpose of this section:
-------------------------
Describe the pre-processing. Nothing is fitted yet - this cell only builds the
recipe that the Pipeline will follow later, on the training data.
"""

# Columns that are genuine numbers -> fill gaps with the median, then rescale
NUMERIC_FEATURES = ["thalach", "oldpeak"]

# Columns that are category codes -> fill gaps with the most common code, then one-hot encode
CATEGORICAL_FEATURES = ["cp", "exang"]

numeric_steps = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_steps = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # handle_unknown="ignore" means an unexpected code at prediction time
    # becomes all-zeros instead of crashing the app
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_steps, NUMERIC_FEATURES),
    ("categorical", categorical_steps, CATEGORICAL_FEATURES),
])

print("\n\nPre-processing Recipe (not fitted yet):")
print("=======================================\n")
print(preprocessor)




Pre-processing Recipe (not fitted yet):

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['thalach', 'oldpeak']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['cp', 'exang'])])


### Step 6: Execute the Training Phase


#### Step 6.1: Split into Training Data and Testing Data

The data is split 80% / 20%:

* **Training data** teaches the model.
* **Testing data** is held out and never seen during training, so the score
  measured on it is an honest estimate of real-world performance.

Two differences from the Titanic notebook, both deliberate:

* `shuffle=True` (the default): the Cleveland file is **not** randomly ordered,
  so taking the last 20% of rows unshuffled would give a biased test set.
* `stratify=y`: keeps the same proportion of disease / no-disease cases in both
  the training and the testing split.


In [8]:
"""
Purpose of this section:
-------------------------
Split the inputs (X) and the output (y) into a training set (80%) and a testing
set (20%). random_state=0 makes the split reproducible - re-running the notebook
gives the same split and therefore the same scores.
"""

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    shuffle=True,
    stratify=y,
)

print("\n\nSplit Sizes:")
print("============\n")
print("Training inputs  (X_train):", X_train.shape)
print("Testing inputs   (X_test) :", X_test.shape)
print("Training labels  (y_train):", y_train.shape)
print("Testing labels   (y_test) :", y_test.shape)

print("\n\nClass Balance After the Stratified Split:")
print("=========================================\n")
print("Training:", y_train.value_counts().sort_index().to_dict())
print("Testing :", y_test.value_counts().sort_index().to_dict())




Split Sizes:

Training inputs  (X_train): (242, 4)
Testing inputs   (X_test) : (61, 4)
Training labels  (y_train): (242,)
Testing labels   (y_test) : (61,)


Class Balance After the Stratified Split:

Training: {0: 131, 1: 111}
Testing : {0: 33, 1: 28}


### Step 6.2: Input Vectors and Labels of the Training Data

`train_test_split` already returned the inputs and the labels separately, so
there is nothing left to separate. This cell just shows what the training input
vectors look like.


In [9]:
"""
Purpose of this section:
-------------------------
Show the training input vectors and the matching training labels.
"""

print("\n\nTraining Input Vectors (first 5 rows):")
print("======================================\n")
print(X_train.head())

print("\n\nMatching Training Labels (first 5):")
print("===================================\n")
print(y_train.head().to_list())




Training Input Vectors (first 5 rows):

     cp  thalach  exang  oldpeak
111   4      144      1      1.2
52    4      153      0      0.0
173   4      157      0      1.2
138   4      130      1      1.6
11    2      153      0      1.3


Matching Training Labels (first 5):

[1, 1, 0, 1, 0]


### Step 6.3: Train the Support Vector Classifier (inside the Pipeline)

The pre-processing recipe from Step 5 and the Support Vector Classifier are
joined into a **single `Pipeline` object**. Calling `.fit(X_train, y_train)` on
it does everything in order:

1. fit the imputers on the training data and fill any gaps,
2. fit the scaler on the training data and rescale the numeric columns,
3. fit the one-hot encoder on the training data and expand the category columns,
4. train the SVC on the result.

Because every `fit` sees **only `X_train`**, the test set stays genuinely unseen.


In [10]:
"""
Purpose of this section:
-------------------------
Build the full Pipeline (pre-processing + SVC) and train it on the training data.
"""

print("\n\nTraining the Support Vector Classifier on the Training Data")
print("===========================================================\n")

heart_disease_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", svm.SVC(kernel="rbf", gamma="scale", random_state=0)),
])

# One call fits the imputers, the scaler, the encoder AND the SVC - training data only
heart_disease_pipeline.fit(X_train, y_train)

print(heart_disease_pipeline)

print("\n\nColumns produced by the pre-processing step:")
print("============================================\n")
print(list(heart_disease_pipeline.named_steps["preprocessor"].get_feature_names_out()))
print("\n(4 original attributes -> the expanded columns the SVC actually sees)")




Training the Support Vector Classifier on the Training Data

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['thalach', 'oldpeak']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknow

### Step 6.4: Save the Trained Pipeline

The assignment requires the model **and** its pre-processing steps to be saved
together in **one** `.pkl` file. Because everything is inside a single
`Pipeline` object, saving that one object is enough - the Streamlit app then
cannot possibly pre-process a user's input differently from the training data.


In [11]:
"""
Purpose of this section:
-------------------------
Save the whole trained Pipeline (imputers + scaler + encoder + SVC) into one
file so the Streamlit app can load it without retraining.
"""

MODEL_FILE = "heart_disease_model.pkl"

pickle.dump(heart_disease_pipeline, open(MODEL_FILE, "wb"))

print("Saved:", MODEL_FILE)
print("This single file contains the imputers, the scaler, the one-hot encoder")
print("and the trained Support Vector Classifier.")


Saved: heart_disease_model.pkl
This single file contains the imputers, the scaler, the one-hot encoder
and the trained Support Vector Classifier.


### Step 7: Execute the Testing Phase


#### Step 7.1: Input Vectors and Labels of the Testing Data

`X_test` and `y_test` were produced by the split in Step 6.1 and have not been
touched since. This cell simply confirms what they contain.


In [12]:
"""
Purpose of this section:
-------------------------
Show the held-out testing input vectors and their true labels.
"""

print("\n\nTesting Input Vectors (first 5 rows):")
print("=====================================\n")
print(X_test.head())

print("\n\nTrue Testing Labels (first 5):")
print("==============================\n")
print(y_test.head().to_list())

print("\nNumber of held-out test patients:", len(X_test))




Testing Input Vectors (first 5 rows):

     cp  thalach  exang  oldpeak
125   2      175      0      0.6
83    3      150      1      1.6
53    2      188      0      0.0
45    3      165      0      2.5
248   4      168      0      1.0


True Testing Labels (first 5):

[0, 1, 0, 1, 1]

Number of held-out test patients: 61


#### Step 7.2: Load the Saved Pipeline


In [13]:
"""
Purpose of this section:
-------------------------
Load the Pipeline back from disk. Doing this proves the saved file really works
and is exactly what the Streamlit app will do.
"""

loaded_model = pickle.load(open(MODEL_FILE, "rb"))

print("Model loaded from", MODEL_FILE)
print(loaded_model)


Model loaded from heart_disease_model.pkl


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['thalach', 'oldpeak']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                               

#### Step 7.3: Make Predictions on the Testing Data


In [14]:
"""
Purpose of this section:
-------------------------
Ask the loaded Pipeline to predict the outcome for every held-out test patient,
then put the true answers and the predictions side by side.
"""

# The raw, unprocessed test inputs go straight in - the Pipeline pre-processes them
model_predictions = loaded_model.predict(X_test)

results = X_test.copy()
results["Actual"] = y_test.values
results["Predicted"] = model_predictions

print("\n\nPredictions Returned by heart_disease_model (first 15 test patients):")
print("=====================================================================\n")
print(results.head(15))




Predictions Returned by heart_disease_model (first 15 test patients):

     cp  thalach  exang  oldpeak  Actual  Predicted
125   2      175      0      0.6       0          0
83    3      150      1      1.6       1          0
53    2      188      0      0.0       0          0
45    3      165      0      2.5       1          0
248   4      168      0      1.0       1          0
211   1      182      1      3.8       1          0
36    4      120      1      2.5       1          1
106   4      162      1      0.0       1          1
120   4      150      1      0.0       1          1
31    4      160      1      1.4       1          1
156   4      173      1      1.6       1          1
157   4      171      0      0.0       1          0
148   2      170      0      0.0       0          0
253   3      157      0      0.6       0          0
250   4      126      1      1.5       0          1


#### Step 7.4: Calculate the Accuracy Score


In [15]:
"""
Purpose of this section:
-------------------------
Accuracy = the share of held-out test patients the model classified correctly.
"""

model_accuracy_score = accuracy_score(y_test, model_predictions)

print("\n\nAccuracy Score on the Held-Out Test Set:")
print("========================================\n")
print(round(model_accuracy_score, 4), " (", round(model_accuracy_score * 100, 2), "% )", sep="")




Accuracy Score on the Held-Out Test Set:

0.8033 (80.33% )


#### Step 7.5: Detailed Metrics (Precision, Recall, F1, Confusion Matrix)

Accuracy on its own hides *what kind* of mistakes the model makes, which matters
a lot in a medical setting:

* **Precision** (disease class): of all the patients flagged as having disease,
  how many really do?
* **Recall** (disease class): of all the patients who really have disease, how
  many did the model catch? In medicine this is usually the most important
  number - a missed case is worse than a false alarm.
* **F1-score**: the harmonic mean of precision and recall.
* **Confusion Matrix**: the full table of correct and incorrect predictions.


In [16]:
"""
Purpose of this section:
-------------------------
Print the full classification report and the confusion matrix.
"""

print("\nClassification Report:")
print("======================\n")
print(classification_report(
    y_test,
    model_predictions,
    target_names=["No Disease (0)", "Disease (1)"],
    digits=4,
))

conf_matrix = confusion_matrix(y_test, model_predictions)
conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=["Actual: No Disease", "Actual: Disease"],
    columns=["Predicted: No Disease", "Predicted: Disease"],
)

print("\nConfusion Matrix:")
print("=================\n")
print(conf_matrix_df)

true_negative, false_positive, false_negative, true_positive = conf_matrix.ravel()
print("\nReading the matrix:")
print(f"  True Negatives  (healthy, correctly cleared) : {true_negative}")
print(f"  False Positives (healthy, wrongly flagged)   : {false_positive}")
print(f"  False Negatives (ill, missed by the model)   : {false_negative}")
print(f"  True Positives  (ill, correctly flagged)     : {true_positive}")



Classification Report:



                precision    recall  f1-score   support

No Disease (0)     0.7442    0.9697    0.8421        33
   Disease (1)     0.9444    0.6071    0.7391        28

      accuracy                         0.8033        61
     macro avg     0.8443    0.7884    0.7906        61
  weighted avg     0.8361    0.8033    0.7948        61




Confusion Matrix:

                    Predicted: No Disease  Predicted: Disease
Actual: No Disease                     32                   1
Actual: Disease                        11                  17

Reading the matrix:
  True Negatives  (healthy, correctly cleared) : 32
  False Positives (healthy, wrongly flagged)   : 1
  False Negatives (ill, missed by the model)   : 11
  True Positives  (ill, correctly flagged)     : 17


### Step 8: Execute the Application Phase


#### Step 8.1: Take Input from the User

Running this cell in an interactive Jupyter session would normally prompt with
`input()`. So that the notebook can also be run top-to-bottom without typing,
a demo patient is pre-filled below.

To try your own values interactively, replace the four lines with:

```python
cp_input      = int(input("Chest pain type (1=typical angina, 2=atypical, 3=non-anginal, 4=asymptomatic): "))
thalach_input = int(input("Maximum heart rate achieved (70-205): "))
exang_input   = int(input("Exercise-induced angina (0=no, 1=yes): "))
oldpeak_input = float(input("ST depression induced by exercise (0.0-6.2): "))
```


In [17]:
"""
Purpose of this section:
-------------------------
Collect the FOUR attribute values for one new, unseen patient.
"""

# Demo patient (edit these values, or swap in the input() calls shown above)
cp_input      = 4     # asymptomatic chest pain
thalach_input = 120   # maximum heart rate achieved
exang_input   = 1     # exercise-induced angina: yes
oldpeak_input = 2.5   # ST depression

print("User input collected:")
print(f"  cp      = {cp_input}")
print(f"  thalach = {thalach_input}")
print(f"  exang   = {exang_input}")
print(f"  oldpeak = {oldpeak_input}")


User input collected:
  cp      = 4
  thalach = 120
  exang   = 1
  oldpeak = 2.5


#### Step 8.2: Convert the User Input into a Feature Vector

The DataFrame must have the **same four column names in the same order** as the
data the Pipeline was trained on. No manual encoding or scaling is done here -
the Pipeline handles all of that itself.


In [18]:
"""
Purpose of this section:
-------------------------
Turn the four values into a one-row DataFrame shaped exactly like the training data.
"""

user_input = pd.DataFrame({
    "cp":      [cp_input],
    "thalach": [thalach_input],
    "exang":   [exang_input],
    "oldpeak": [oldpeak_input],
})

# Guarantee the column order matches the training data
user_input = user_input[SELECTED_FEATURES]

print("\n\nUser Input Feature Vector:")
print("==========================\n")
print(user_input)




User Input Feature Vector:

   cp  thalach  exang  oldpeak
0   4      120      1      2.5


#### Step 8.3: Load the Saved Pipeline

Loaded again here so that the Application Phase can be run on its own, exactly
the way `app.py` does it.


In [19]:
"""
Purpose of this section:
-------------------------
Load the saved Pipeline from the .pkl file, ready to predict.
"""

model = pickle.load(open(MODEL_FILE, "rb"))

print("Model loaded from", MODEL_FILE)


Model loaded from heart_disease_model.pkl


#### Step 8.4: Model Prediction


In [20]:
"""
Purpose of this section:
-------------------------
Run the user's feature vector through the Pipeline and report the outcome.
The Pipeline imputes, scales and encodes the input automatically, using the
values it learned from the TRAINING data, before the SVC sees it.
"""

predicted_outcome = int(model.predict(user_input)[0])

if predicted_outcome == 1:
    prediction = "DISEASE INDICATED"
else:
    prediction = "NO DISEASE INDICATED"

print("\n" + "=" * 46)
print("**            Prediction                   **")
print("=" * 46)
print(f"   {prediction}")
print("=" * 46)

print("\nDisclaimer: this is an educational prediction system built for a")
print("machine learning assignment. It is NOT a medical diagnosis. Always")
print("consult a qualified healthcare professional.")



**            Prediction                   **
   DISEASE INDICATED

Disclaimer: this is an educational prediction system built for a
machine learning assignment. It is NOT a medical diagnosis. Always
consult a qualified healthcare professional.


### Step 9: Execute the Feedback Phase


#### Step 9.1: Collect Feedback from Users and Domain Experts


In [21]:
"""
Purpose of this section:
-------------------------
Step 9.1: Example feedback on the model's behaviour, of the kind that would be
collected after showing the predictions to a domain expert and to testers.
"""

example_feedback = pd.DataFrame({
    "Reviewer": ["Domain Expert (Cardiology)", "Domain Expert (Cardiology)", "Student Tester"],
    "Case": [
        "Asymptomatic chest pain + exercise angina + high ST depression -> DISEASE INDICATED",
        "Several true disease cases were predicted as NO DISEASE (see the false negatives in Step 7.5)",
        "Changing only `thalach` by a few beats per minute rarely changes the answer",
    ],
    "Feedback": [
        "Clinically sensible - this combination is a textbook high-risk presentation.",
        "The main weakness. Missing a real case is more costly than a false alarm, so recall on the disease class should be raised even if overall accuracy drops.",
        "Expected and desirable - the model should not be hypersensitive to small measurement noise.",
    ],
})

print(example_feedback.to_string(index=False))


                  Reviewer                                                                                          Case                                                                                                                                                  Feedback
Domain Expert (Cardiology)           Asymptomatic chest pain + exercise angina + high ST depression -> DISEASE INDICATED                                                                              Clinically sensible - this combination is a textbook high-risk presentation.
Domain Expert (Cardiology) Several true disease cases were predicted as NO DISEASE (see the false negatives in Step 7.5) The main weakness. Missing a real case is more costly than a false alarm, so recall on the disease class should be raised even if overall accuracy drops.
            Student Tester                   Changing only `thalach` by a few beats per minute rarely changes the answer                                                       

#### Step 9.2: Make a List of Potential Improvements


In [22]:
"""
Purpose of this section:
-------------------------
Step 9.2: Concrete, specific improvements suggested by the feedback above and
by the metrics in Step 7.5.
"""

improvements = [
    "Raise recall on the disease class: pass class_weight='balanced' to the SVC, or train with probability=True and lower the decision threshold below 0.5, so fewer real cases are missed.",
    "Tune the SVC hyper-parameters C and gamma with GridSearchCV instead of relying on the defaults.",
    "Report cross-validated scores (e.g. 5-fold StratifiedKFold) instead of a single 80/20 split - with only 303 rows, one split is noisy and the accuracy can shift by several points with a different random_state.",
    "Use more of the dataset: the four selected attributes are a deliberate assignment constraint, and adding ca, thal and slope is known to push accuracy higher.",
    "Add the other three UCI subsets (Hungarian, Switzerland, Long Beach VA, ~920 patients in total) so the model is not fitted to one hospital's population.",
    "Compare against other classifiers (Logistic Regression, Random Forest, Gradient Boosting) to confirm SVC is actually the right choice here.",
    "Show a confidence score in the app rather than a bare yes/no, so a borderline case is visibly borderline.",
]

for number, item in enumerate(improvements, 1):
    print(f"{number}. {item}\n")


1. Raise recall on the disease class: pass class_weight='balanced' to the SVC, or train with probability=True and lower the decision threshold below 0.5, so fewer real cases are missed.

2. Tune the SVC hyper-parameters C and gamma with GridSearchCV instead of relying on the defaults.

3. Report cross-validated scores (e.g. 5-fold StratifiedKFold) instead of a single 80/20 split - with only 303 rows, one split is noisy and the accuracy can shift by several points with a different random_state.

4. Use more of the dataset: the four selected attributes are a deliberate assignment constraint, and adding ca, thal and slope is known to push accuracy higher.

5. Add the other three UCI subsets (Hungarian, Switzerland, Long Beach VA, ~920 patients in total) so the model is not fitted to one hospital's population.

6. Compare against other classifiers (Logistic Regression, Random Forest, Gradient Boosting) to confirm SVC is actually the right choice here.

7. Show a confidence score in the app

#### Step 9.3: Improve the Model Based on Feedback


In [23]:
"""
Purpose of this section:
-------------------------
Step 9.3: In a real deployment the next version would apply the improvements
above and re-run Steps 6-7, only replacing the deployed .pkl file if the new
model genuinely scores better on the held-out test data.

The deployed model is intentionally left as the simple version so that this
notebook stays a clear, beginner-friendly match for the Titanic reference.
"""

print("Current deployed model : SVC (RBF kernel, default C and gamma) on 4 attributes")
print("Next version           : class_weight='balanced' + GridSearchCV over C and gamma,")
print("                         validated with 5-fold cross-validation before replacing")
print("                        ", MODEL_FILE)


Current deployed model : SVC (RBF kernel, default C and gamma) on 4 attributes
Next version           : class_weight='balanced' + GridSearchCV over C and gamma,
                         validated with 5-fold cross-validation before replacing
                         heart_disease_model.pkl


## Step 10: Cloud Deployment Readiness

The trained model is saved in a form that a web app can use directly, so the
project is ready to run as a Streamlit application rather than only inside this
notebook.

#### Step 10.1: The Files That Make Up the Project

| File | Role |
| ---- | ---- |
| `heart.csv` | the UCI Cleveland dataset |
| `Heart_Disease_Prediction.ipynb` | this notebook - trains and saves the model |
| `heart_disease_model.pkl` | the saved Pipeline (pre-processing + SVC) |
| `app.py` | the Streamlit web app |
| `requirements.txt` | the exact package versions the app needs |

#### Step 10.2: How the App Reuses This Notebook's Work

`app.py` does not retrain anything. It loads `heart_disease_model.pkl` and calls
`.predict()` on a one-row DataFrame holding the same four attributes, in the same
order, that Step 4 selected.

Because the imputers, the scaler and the one-hot encoder are stored inside the
same Pipeline object as the classifier, the app cannot pre-process a user's input
differently from the training data. That is the practical reason Step 6.4 saved
one combined `.pkl` rather than separate model and encoder files.

#### Step 10.3: Why `requirements.txt` Pins Exact Versions

`heart_disease_model.pkl` is a pickled scikit-learn object. Unpickling it with a
different scikit-learn version than the one that created it can produce warnings
or outright failures. Pinning the versions means the deployment environment
matches the one this notebook was run in, so the saved model loads cleanly.

If the model is ever retrained with newer packages, `requirements.txt` has to be
updated to match, and both files replaced together.


## Conclusion

This notebook built a Heart Disease Prediction System by following the same
end-to-end workflow as the Titanic reference notebook, applied to the authentic
UCI Cleveland heart disease dataset (303 patients).

The original UCI target `num` (0-4) was converted into a binary target, and
**exactly four** input attributes were used: chest pain type (`cp`), maximum
heart rate achieved (`thalach`), exercise-induced angina (`exang`) and
exercise-induced ST depression (`oldpeak`). Together these cover a reported
symptom, a performance measurement, an observed symptom under load and an
objective instrument reading - four different kinds of clinical evidence.

A Support Vector Classifier was trained inside a scikit-learn `Pipeline` that
also holds the imputers, the scaler and the one-hot encoder. Everything was
fitted on the training split only, so the score on the held-out 20% is an honest
estimate, and the whole pipeline was saved to a single `heart_disease_model.pkl`
that `app.py` loads directly. The exact accuracy, precision, recall, F1-score and
confusion matrix are printed by Steps 7.4 and 7.5 above.

The clearest weakness is recall on the disease class: the model is noticeably
better at clearing healthy patients than at catching ill ones, which is the wrong
way round for a medical screening tool. Step 9.2 lists the specific changes that
would address this.

**Disclaimer:** this is an educational project for a machine learning assignment.
It is not a medical device and it must not be used for diagnosis. Anyone with
health concerns should consult a qualified healthcare professional.
